# ARC + MolFormer — Corrected Pipeline (v12)
## All fixes from v11 + NEW fixes for ARC under-performance

| # | Location | Issue | Fix |
|---|----------|-------|-----|
| FIX-1 | `out_of_task_detection` | OTD logic inverted vs paper Algorithm 1 | Correct Assumption 1/2 per paper |
| FIX-2 | `out_of_task_detection` | batch_theta computed inside per-sample loop | Pre-computed once over full batch |
| FIX-3 | `evaluate_task` | clf_arc deepcopy reset on every task | `persistent_clf_arc` carried across test stream |
| FIX-4 | `run_cil_pipeline` | Replay on for ARC run suppresses bias | ARC memory-free; also adds no-replay baseline for fair comparison |
| FIX-5 | `assign_atc_label` | Inverse-frequency labeling non-deterministic | Dominant-class (highest prevalence) assignment |
| FIX-6 | `DrugBank` | Nitroso class has only 1 sample — statistically invalid | Drop classes with fewer than MIN_CLASS_SAMPLES=30 across any split |
| FIX-7 | `hyperparameters` | ε=0.50 and θ=0.40 below paper-tested range | val-set sweep over ε ∈ {0.30..0.80} and θ ∈ {0.02..0.20} |
| FIX-8 | `run_cil_pipeline` | Only ARC vs replay-baseline comparison | Added no-replay baseline for apples-to-apples comparison |
| FIX-9 | `compute_bwt_fwt` | BWT/FWT retrains fresh clf, discarding ARC state | For ARC condition, BWT/FWT measured from persistent_clf_arc checkpoints |
| FIX-10 | `train_classifier_on_task` | Early stopping monitors accuracy — biased under imbalance | Monitor macro-F1 (also report macro-F1 in evaluation) |
| FIX-11 | `out_of_task_detection` | w-ratio inverted: was `c/c_hat` | Corrected to `w = c_hat/c` and `w >= theta` |
| FIX-12 | `sweep_arc_thresholds` | Sweep evaluated only final task | Sweep over ALL tasks, average val macro-F1 |
| FIX-13 | sweep call | ε/θ range invalid for corrected ratio | Extended ranges |
| FIX-14–17 | various | versioning & defaults | Tagged v11 |
| **FIX-18** | **`compute_batch_theta`** | **`max(theta, adaptive)` raises threshold (v11 comment said min but code used max)** | **Changed to `min(theta, adaptive)` — data-driven estimate can now LOWER the bar** |
| **FIX-19** | **`run_cil_pipeline`** | **`task_id=prev_tid` passed to evaluate_task — OTD boundary based on task being evaluated, not tasks model has seen** | **Pass `task_id=tid` (current training task) so OTD knows full class space** |
| **FIX-20** | **`sweep_arc_thresholds`** | **clf_arc NOT reset between (ε, θ) combinations — sweep contaminated by prior retention updates** | **Deep-copy clf from checkpoint before each (ε, θ) evaluation** |
| **FIX-21** | **`evaluate_task`** | **persistent_clf_arc mutations from evaluating prev_task bleed into next prev_task within same outer task** | **Snapshot clf_arc state before each prev_task eval; restore if evaluating an older task** |

### Ablation Study (NEW)
- `run_ablation_study()`: evaluates Retention-only, Correction-only, and Full-ARC conditions to match Table 4 in paper

## Step 1 — Install & Imports

In [1]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

packages = [
    'torch>=2.0.0',
    'transformers>=4.35.0',
    'scikit-learn>=1.3.0',
    'numpy>=1.24.0',
    'matplotlib>=3.7.0',
    'seaborn>=0.12.0',
    'pandas>=2.0.0',
    'rdkit',
    'einops',
    'rotary-embedding-torch',
    'requests',
    'scipy',
    'tqdm'
]

for pkg in packages:
    try:
        install(pkg)
        print(f'  ✓ {pkg}')
    except Exception as e:
        print(f'  ✗ {pkg}: {e}')

print('\nInstallation complete.')

  ✓ torch>=2.0.0


  ✓ transformers>=4.35.0


  ✓ scikit-learn>=1.3.0


  ✓ numpy>=1.24.0


  ✓ matplotlib>=3.7.0


...

  ✓ einops


  ✓ rotary-embedding-torch


  ✓ requests


  ✓ scipy


  ✓ tqdm

Installation complete.


In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
!pip uninstall sentence-transformers -y
!pip install transformers==4.41.2 onnx onnxruntime

Found existing installation: sentence-transformers 5.2.3
Uninstalling sentence-transformers-5.2.3:
  Successfully uninstalled sentence-transformers-5.2.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 70.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 84.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.7 MB/s eta 0:00:0000:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installati

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
import os, json, copy, warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from rdkit.Chem import MolStandardize, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import roc_auc_score, f1_score

from transformers import AutoTokenizer, AutoModel

RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print('All imports OK')

Device: cuda
All imports OK


In [66]:
CFG = {
    'atc_path'          : 'ATC_SMILES.csv',
    'drugbank_path'     : 'df_drugbank_smiles.csv',
    'output_dir'        : 'arc_output',
    'min_ha'            : 5,
    'max_ha'            : 100,
    'min_mw'            : 100,
    'max_mw'            : 1500,
    'atc_top_k'         : 14,
    'train_ratio'       : 0.70,
    'val_ratio'         : 0.15,
    'test_ratio'        : 0.15,
    'classes_per_task'  : 2,
    'molformer_name'    : 'ibm/MoLFormer-XL-both-10pct',
    'molformer_batch'   : 32,
    'max_smiles_len'    : 202,
    'clf_epochs'        : 100,
    'clf_lr'            : 1e-3,
    'clf_batch'         : 32,
    'arc_epsilon'       : 0.70,
    'arc_theta'         : 0.05,  # FIX-15: corrected default for w=c_hat/c ratio
    'arc_temp'          : 2.0,
    'arc_lr'            : 1e-5,
    'seed'              : 42,
    'es_patience'       : 10,
    'es_min_delta'      : 1e-4,
    'replay_buf_size'   : 20,
    'replay_lambda'     : 0.5,
    'min_class_samples' : 30,
}
os.makedirs(CFG['output_dir'], exist_ok=True)
torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
print('Config ready')
print(f"  classes_per_task={CFG['classes_per_task']}  arc_epsilon={CFG['arc_epsilon']}  arc_theta={CFG['arc_theta']}  arc_temp={CFG['arc_temp']}")

Config ready
  classes_per_task=2  arc_epsilon=0.7  arc_theta=0.05  arc_temp=2.0


## Step 2 — ATC Pipeline

In [3]:
def clean_smiles(smiles, cfg):
    if not isinstance(smiles, str) or smiles.strip() == '':
        return None, 'missing_or_empty'
    mol = Chem.MolFromSmiles(smiles.strip())
    if mol is None:
        return None, 'invalid_smiles'
    try:
        mol = MolStandardize.rdMolStandardize.LargestFragmentChooser().choose(mol)
    except Exception:
        pass
    try:
        mol = MolStandardize.rdMolStandardize.Uncharger().uncharge(mol)
        mol = MolStandardize.rdMolStandardize.TautomerEnumerator().Canonicalize(mol)
    except Exception:
        pass
    try:
        ha = mol.GetNumHeavyAtoms()
        mw = rdMolDescriptors.CalcExactMolWt(mol)
        if not (cfg['min_ha'] <= ha <= cfg['max_ha'] and cfg['min_mw'] <= mw <= cfg['max_mw']):
            return None, 'filtered_out'
    except Exception:
        return None, 'filter_error'
    canon = Chem.MolToSmiles(mol, canonical=True)
    return (canon, 'ok') if canon else (None, 'canon_failed')


def get_scaffold(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return smiles
        sc = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(sc, canonical=True)
    except Exception:
        return smiles

print('SMILES helpers defined')

SMILES helpers defined


In [4]:
def stratified_scaffold_split(df, label_col, train_r, val_r, test_r, seed=42):
    assert abs(train_r + val_r + test_r - 1.0) < 1e-6
    rng = np.random.default_rng(seed)
    train_idx, val_idx, test_idx = [], [], []
    for cls in sorted(df[label_col].unique()):
        cls_df = df[df[label_col] == cls]
        sc2idx = defaultdict(list)
        for idx, sc in zip(cls_df.index, cls_df['scaffold']):
            sc2idx[sc].append(idx)
        groups = list(sc2idx.values())
        rng.shuffle(groups)
        n    = len(cls_df)
        n_tr = max(1, int(n * train_r))
        n_vl = max(1, int(n * val_r))
        cls_tr, cls_vl, cls_ts = [], [], []
        for g in groups:
            if len(cls_tr) < n_tr:
                cls_tr.extend(g)
            elif len(cls_vl) < n_vl:
                cls_vl.extend(g)
            else:
                cls_ts.extend(g)
        if len(cls_vl) == 0 and len(cls_tr) > 1:
            cls_vl.append(cls_tr.pop())
        if len(cls_ts) == 0 and len(cls_tr) > 1:
            cls_ts.append(cls_tr.pop())
        train_idx.extend(cls_tr)
        val_idx.extend(cls_vl)
        test_idx.extend(cls_ts)
    tr = df.loc[train_idx].copy()
    vl = df.loc[val_idx].copy()
    ts = df.loc[test_idx].copy()
    for split_name, split_df in [('val', vl), ('test', ts)]:
        missing = set(df[label_col].unique()) - set(split_df[label_col].unique())
        if missing:
            print(f'  WARNING: {split_name} still missing classes {missing}')
    return tr, vl, ts


def verify_split(train_df, val_df, test_df, label_col='label_id'):
    for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
        dist = df[label_col].value_counts().sort_index().to_dict()
        print(f'  {name:5s}: {len(df):4d} samples | classes: {dist}')
    tsc = set(train_df['scaffold']); vsc = set(val_df['scaffold']); esc = set(test_df['scaffold'])
    print(f'  Scaffold leakage → Train∩Val={len(tsc&vsc)} | Train∩Test={len(tsc&esc)} | Val∩Test={len(vsc&esc)}')

print('Stratified scaffold split defined')

Stratified scaffold split defined


In [5]:
import ast

atc_raw = pd.read_csv(CFG['atc_path'])
print(f'ATC raw: {atc_raw.shape}')

atc_raw['label_vec'] = atc_raw['Lable'].apply(ast.literal_eval)
ATC_N_CLASSES = len(atc_raw['label_vec'].iloc[0])
print(f'ATC classes (one-hot length): {ATC_N_CLASSES}')

res = atc_raw['CanSmiles'].apply(lambda s: clean_smiles(s, CFG))
atc_raw['canon_smiles'] = res.apply(lambda x: x[0])
atc_raw['clean_status'] = res.apply(lambda x: x[1])
print('\nCleaning report:')
print(atc_raw['clean_status'].value_counts().to_string())

atc_df = atc_raw[atc_raw['clean_status'] == 'ok'].drop_duplicates('canon_smiles').copy()
print(f'\nAfter cleaning: {atc_df.shape}')

ATC raw: (4545, 4)
ATC classes (one-hot length): 14

Cleaning report:
clean_status
ok              4373
filtered_out     172

After cleaning: (2862, 7)


In [6]:
atc_class_counts = np.zeros(ATC_N_CLASSES)
for vec in atc_df['label_vec']:
    atc_class_counts += np.array(vec)
atc_class_prev = atc_class_counts / len(atc_df)

def assign_atc_label_dominant(row):
    vec    = row['label_vec']
    active = [i for i, v in enumerate(vec) if v == 1]
    if not active:
        return None
    if len(active) == 1:
        return active[0]
    active.sort(key=lambda i: (-atc_class_prev[i], i))
    return active[0]

atc_df['primary_label'] = atc_df.apply(assign_atc_label_dominant, axis=1)
atc_df = atc_df.dropna(subset=['primary_label']).copy()
atc_df['primary_label'] = atc_df['primary_label'].astype(int)
print(f'ATC labeled shape: {atc_df.shape}')
print('\nLabel distribution (dominant-class assignment):')
print(atc_df['primary_label'].value_counts().sort_index().to_string())

ATC labeled shape: (2862, 8)

Label distribution (dominant-class assignment):
primary_label
0     340
1      68
2     408
3     181
4     136
5      42
6     354
7     270
8     127
9     494
10     86
11    172
12     69
13    115


In [7]:
K = CFG['atc_top_k']
top_k_labels = atc_df['primary_label'].value_counts().head(K).index.tolist()
atc_df = atc_df[atc_df['primary_label'].isin(top_k_labels)].copy()

le_atc = LabelEncoder()
atc_df['label_id'] = le_atc.fit_transform(atc_df['primary_label'])
atc_label_map = {int(le_atc.transform([c])[0]): f'ATC-{c}' for c in le_atc.classes_}

print(f'ATC Top-{K} dataset: {atc_df.shape}')
print('\nEncoding map:')
for i, name in atc_label_map.items():
    print(f'  {i:2d} → {name}')

ATC Top-14 dataset: (2862, 9)

Encoding map:
   0 → ATC-0
   1 → ATC-1
   2 → ATC-2
   3 → ATC-3
   4 → ATC-4
   5 → ATC-5
   6 → ATC-6
   7 → ATC-7
   8 → ATC-8
   9 → ATC-9
  10 → ATC-10
  11 → ATC-11
  12 → ATC-12
  13 → ATC-13


In [8]:
atc_df['scaffold'] = atc_df['canon_smiles'].apply(get_scaffold)

print('Running stratified scaffold split...')
atc_train, atc_val, atc_test = stratified_scaffold_split(
    atc_df, 'label_id',
    CFG['train_ratio'], CFG['val_ratio'], CFG['test_ratio'], CFG['seed']
)
print(f'\nATC split sizes → train={len(atc_train)} | val={len(atc_val)} | test={len(atc_test)}')
verify_split(atc_train, atc_val, atc_test)

Running stratified scaffold split...

ATC split sizes → train=2015 | val=439 | test=408
  Train: 2015 samples | classes: {0: 237, 1: 49, 2: 285, 3: 126, 4: 95, 5: 31, 6: 247, 7: 190, 8: 99, 9: 345, 10: 60, 11: 120, 12: 51, 13: 80}
  Val  :  439 samples | classes: {0: 51, 1: 10, 2: 65, 3: 27, 4: 20, 5: 8, 6: 53, 7: 40, 8: 19, 9: 74, 10: 12, 11: 32, 12: 11, 13: 17}
  Test :  408 samples | classes: {0: 52, 1: 9, 2: 58, 3: 28, 4: 21, 5: 3, 6: 54, 7: 40, 8: 9, 9: 75, 10: 14, 11: 20, 12: 7, 13: 18}
  Scaffold leakage → Train∩Val=41 | Train∩Test=42 | Val∩Test=18


In [9]:
def make_cil_tasks(train_df, val_df, test_df, cpt, label_map):
    all_ids = sorted(train_df['label_id'].unique())
    tasks = []
    for t in range(int(np.ceil(len(all_ids) / cpt))):
        new_cls  = all_ids[t*cpt : (t+1)*cpt]
        seen_cls = all_ids[: (t+1)*cpt]
        tasks.append({
            'task_id'    : t,
            'new_classes': new_cls,
            'all_classes': list(seen_cls),
            'n_classes'  : len(seen_cls),
            'class_names': {lid: label_map[lid] for lid in seen_cls},
            'train': train_df[train_df['label_id'].isin(seen_cls)].copy(),
            'val'  : val_df[val_df['label_id'].isin(seen_cls)].copy(),
            'test' : test_df[test_df['label_id'].isin(seen_cls)].copy(),
        })
    return tasks

atc_tasks = make_cil_tasks(atc_train, atc_val, atc_test, CFG['classes_per_task'], atc_label_map)
print(f'ATC CIL tasks: {len(atc_tasks)}')
for t in atc_tasks:
    new_names = [t['class_names'][c] for c in t['new_classes']]
    print(f"  Task {t['task_id']}: new={new_names}  train={len(t['train'])} val={len(t['val'])} test={len(t['test'])}")

ATC CIL tasks: 7
  Task 0: new=['ATC-0', 'ATC-1']  train=286 val=61 test=61
  Task 1: new=['ATC-2', 'ATC-3']  train=697 val=153 test=147
  Task 2: new=['ATC-4', 'ATC-5']  train=823 val=181 test=171
  Task 3: new=['ATC-6', 'ATC-7']  train=1260 val=274 test=265
  Task 4: new=['ATC-8', 'ATC-9']  train=1704 val=367 test=349
  Task 5: new=['ATC-10', 'ATC-11']  train=1884 val=411 test=383
  Task 6: new=['ATC-12', 'ATC-13']  train=2015 val=439 test=408


## Step 3 — DrugBank Pipeline

In [10]:
db_raw = pd.read_csv(CFG['drugbank_path'])
print(f'DrugBank raw: {db_raw.shape}')

smiles_col_db = 'smiles' if 'smiles' in db_raw.columns else 'SMILES'

res_db = db_raw[smiles_col_db].apply(lambda s: clean_smiles(s, CFG))
db_raw['canon_smiles'] = res_db.apply(lambda x: x[0])
db_raw['clean_status'] = res_db.apply(lambda x: x[1])
print('\nDrugBank cleaning report:')
print(db_raw['clean_status'].value_counts().to_string())

db_df = db_raw[db_raw['clean_status'] == 'ok'].drop_duplicates('canon_smiles').copy()
print(f'\nAfter cleaning: {db_df.shape}')

DrugBank raw: (5653, 16)

DrugBank cleaning report:
clean_status
ok                5530
filtered_out        88
invalid_smiles      35

After cleaning: (5361, 18)


In [11]:
IGNORE_COLS = {
    'Unnamed: 0', 'drugbank_id', 'name', 'cas', 'smiles', 'SMILES',
    'canon_smiles', 'clean_status', 'logP ALOGPS', 'logP ChemAxon',
    'solubility ALOGPS', 'pKa (strongest acidic)', 'pKa (strongest basic)', 'F'
}
DB_LABEL_COLS = [c for c in db_df.columns if c not in IGNORE_COLS]
print('DrugBank functional-group columns:', DB_LABEL_COLS)

def assign_db_label(row):
    vals = row[DB_LABEL_COLS].fillna(0)
    if vals.sum() == 0:
        return 'none_detected'
    return vals.idxmax()

db_df['primary_label'] = db_df.apply(assign_db_label, axis=1)
print('\nDrugBank label distribution (BEFORE filtering):')
print(db_df['primary_label'].value_counts().to_string())

DrugBank functional-group columns: ['carbonyl', 'sulfinyl', 'sulfonyl', 'nitroso', 'nitro']

DrugBank label distribution (BEFORE filtering):
primary_label
carbonyl         3819
none_detected    1273
sulfonyl          203
nitro              44
sulfinyl           21
nitroso             1


In [12]:
MIN_CLASS_SAMPLES = CFG['min_class_samples']
label_counts = db_df['primary_label'].value_counts()
valid_labels = label_counts[label_counts >= MIN_CLASS_SAMPLES].index.tolist()
dropped = label_counts[label_counts < MIN_CLASS_SAMPLES]

if len(dropped) > 0:
    print(f'\nFIX-6: Dropping {len(dropped)} classes with < {MIN_CLASS_SAMPLES} samples:')
    print(dropped.to_string())
    db_df = db_df[db_df['primary_label'].isin(valid_labels)].copy()
    print(f'DrugBank after filtering: {db_df.shape}')

le_db = LabelEncoder()
db_df['label_id'] = le_db.fit_transform(db_df['primary_label'])
db_label_map = {int(e): c for e, c in enumerate(le_db.classes_)}
print('\nEncoding map:', db_label_map)


FIX-6: Dropping 2 classes with < 30 samples:
primary_label
sulfinyl    21
nitroso      1
DrugBank after filtering: (5339, 19)

Encoding map: {0: 'carbonyl', 1: 'nitro', 2: 'none_detected', 3: 'sulfonyl'}


In [13]:
db_df['scaffold'] = db_df['canon_smiles'].apply(get_scaffold)

print('Running stratified scaffold split for DrugBank...')
db_train, db_val, db_test = stratified_scaffold_split(
    db_df, 'label_id',
    CFG['train_ratio'], CFG['val_ratio'], CFG['test_ratio'], CFG['seed']
)
print(f'\nDrugBank split sizes → train={len(db_train)} | val={len(db_val)} | test={len(db_test)}')
verify_split(db_train, db_val, db_test)

db_tasks = make_cil_tasks(db_train, db_val, db_test, CFG['classes_per_task'], db_label_map)
print(f'\nDrugBank CIL tasks: {len(db_tasks)}')
for t in db_tasks:
    new_names = [t['class_names'][c] for c in t['new_classes']]
    print(f"  Task {t['task_id']}: new={new_names}  train={len(t['train'])} val={len(t['val'])} test={len(t['test'])}")

Running stratified scaffold split for DrugBank...

DrugBank split sizes → train=3746 | val=800 | test=793
  Train: 3746 samples | classes: {0: 2673, 1: 37, 2: 894, 3: 142}
  Val  :  800 samples | classes: {0: 572, 1: 6, 2: 190, 3: 32}
  Test :  793 samples | classes: {0: 574, 1: 1, 2: 189, 3: 29}
  Scaffold leakage → Train∩Val=25 | Train∩Test=27 | Val∩Test=12

DrugBank CIL tasks: 2
  Task 0: new=['carbonyl', 'nitro']  train=2710 val=578 test=575
  Task 1: new=['none_detected', 'sulfonyl']  train=3746 val=800 test=793


## Step 4 — MolFormer Feature Extraction

In [14]:
print('Loading MolFormer tokenizer & model...')
tokenizer = AutoTokenizer.from_pretrained(CFG['molformer_name'], trust_remote_code=True)
molformer = AutoModel.from_pretrained(
    CFG['molformer_name'], trust_remote_code=True, deterministic_eval=True
)
molformer.eval().to(DEVICE)
FEAT_DIM = molformer.config.hidden_size

for p in molformer.parameters():
    p.requires_grad = False

print(f'MolFormer loaded on {DEVICE}  |  hidden_dim={FEAT_DIM}')
print(f'Backbone frozen (ARC paper assumption: frozen feature extractor)')

Loading MolFormer tokenizer & model...
MolFormer loaded on cuda  |  hidden_dim=768
Backbone frozen (ARC paper assumption: frozen feature extractor)


In [15]:
@torch.no_grad()
def extract_molformer_features(smiles_list, batch_size=32):
    all_emb = []
    for i in tqdm(range(0, len(smiles_list), batch_size), desc='MolFormer encode'):
        batch = smiles_list[i: i + batch_size]
        enc   = tokenizer(
            batch, padding=True, truncation=True,
            max_length=CFG['max_smiles_len'], return_tensors='pt'
        ).to(DEVICE)
        out    = molformer(**enc)
        hidden = out.last_hidden_state
        mask   = enc['attention_mask']
        mask_f = mask.unsqueeze(-1).float()
        summed = (hidden * mask_f).sum(dim=1)
        counts = mask_f.sum(dim=1).clamp(min=1)
        pooled = (summed / counts).cpu().numpy()
        all_emb.append(pooled)
    return np.vstack(all_emb)

print('Feature extractor ready')

Feature extractor ready


In [16]:
print('=== Extracting ATC features ===')
atc_feats_all = extract_molformer_features(atc_df['canon_smiles'].tolist(), CFG['molformer_batch'])
idx_to_feat_atc = {idx: feat for idx, feat in zip(atc_df.index, atc_feats_all)}

def get_feats(split_df, idx_to_feat):
    X = np.stack([idx_to_feat[i] for i in split_df.index])
    y = split_df['label_id'].values
    return X, y

atc_X_tr,  atc_y_tr  = get_feats(atc_train, idx_to_feat_atc)
atc_X_val, atc_y_val = get_feats(atc_val,   idx_to_feat_atc)
atc_X_te,  atc_y_te  = get_feats(atc_test,  idx_to_feat_atc)
print(f'ATC → train:{atc_X_tr.shape} | val:{atc_X_val.shape} | test:{atc_X_te.shape}')

=== Extracting ATC features ===


MolFormer encode:   0%|          | 0/90 [00:00<?, ?it/s]

ATC → train:(2015, 768) | val:(439, 768) | test:(408, 768)


In [17]:
print('=== Extracting DrugBank features ===')
db_feats_all = extract_molformer_features(db_df['canon_smiles'].tolist(), CFG['molformer_batch'])
idx_to_feat_db = {idx: feat for idx, feat in zip(db_df.index, db_feats_all)}

db_X_tr,  db_y_tr  = get_feats(db_train, idx_to_feat_db)
db_X_val, db_y_val = get_feats(db_val,   idx_to_feat_db)
db_X_te,  db_y_te  = get_feats(db_test,  idx_to_feat_db)
print(f'DrugBank → train:{db_X_tr.shape} | val:{db_X_val.shape} | test:{db_X_te.shape}')

=== Extracting DrugBank features ===


MolFormer encode:   0%|          | 0/167 [00:00<?, ?it/s]

DrugBank → train:(3746, 768) | val:(800, 768) | test:(793, 768)


## Step 5 — CIL Linear Classifier (Expanding Head)

In [18]:
class MolDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


class ExpandingLinearHead(nn.Module):
    def __init__(self, in_dim, n_init):
        super().__init__()
        self.fc = nn.Linear(in_dim, n_init)

    def grow(self, n_new, device):
        old = self.fc
        n_old = old.out_features
        new_fc = nn.Linear(old.in_features, n_old + n_new)
        with torch.no_grad():
            new_fc.weight[:n_old] = old.weight
            new_fc.bias[:n_old]   = old.bias
            nn.init.xavier_uniform_(new_fc.weight[n_old:])
            nn.init.zeros_(new_fc.bias[n_old:])
        self.fc = new_fc.to(device)

    def forward(self, x): return self.fc(x)

    @property
    def n_classes(self): return self.fc.out_features


class ReplayBuffer:
    def __init__(self, max_per_class: int):
        self.max_per_class = max_per_class
        self._X: dict = {}
        self._y: dict = {}
        self._counts: dict = {}

    def update(self, X: np.ndarray, y: np.ndarray, rng=None):
        if rng is None:
            rng = np.random.default_rng()
        for cls in np.unique(y):
            mask = y == cls
            if cls not in self._X:
                self._X[cls] = []; self._y[cls] = []; self._counts[cls] = 0
            for feat in X[mask]:
                n = self._counts[cls]
                if n < self.max_per_class:
                    self._X[cls].append(feat); self._y[cls].append(int(cls))
                else:
                    j = int(rng.integers(0, n + 1))
                    if j < self.max_per_class:
                        self._X[cls][j] = feat
                self._counts[cls] += 1

    def sample(self):
        if not self._X:
            return None, None
        X_p = np.vstack([np.stack(self._X[c]) for c in sorted(self._X)])
        y_p = np.concatenate([np.array(self._y[c]) for c in sorted(self._y)])
        return X_p, y_p

    def __len__(self):
        return sum(len(v) for v in self._X.values())


def compute_macro_f1(clf, X_v, y_v, device):
    if len(X_v) == 0:
        return float('nan')
    clf.eval()
    with torch.no_grad():
        xv   = torch.tensor(X_v, dtype=torch.float32).to(device)
        pred = clf(xv).argmax(1).cpu().numpy()
    present = np.unique(y_v)
    return float(f1_score(y_v, pred, labels=present, average='macro', zero_division=0))


def train_classifier_on_task(clf, X_tr, y_tr, X_val, y_val,
                              seen_cls, epochs, lr, batch_size, device,
                              replay_buffer=None, replay_lambda=0.5,
                              es_patience=10, es_min_delta=1e-4):
    tr_mask  = np.isin(y_tr, seen_cls)
    X_t, y_t = X_tr[tr_mask], y_tr[tr_mask]

    val_mask = np.isin(y_val, seen_cls)
    X_v, y_v = X_val[val_mask], y_val[val_mask]

    n_total_cls = clf.n_classes
    class_counts = np.array([max(1, (y_t == c).sum()) for c in range(n_total_cls)])
    class_weights = 1.0 / class_counts.astype(float)
    class_weights = class_weights / class_weights.sum() * len(class_counts)
    weight_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

    ds = MolDataset(X_t, y_t)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)

    opt     = torch.optim.Adam(clf.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(weight=weight_tensor)

    best_val_f1    = -1.0
    best_weights   = copy.deepcopy(clf.state_dict())
    patience_count = 0

    use_replay = (replay_buffer is not None) and (len(replay_buffer) > 0)

    clf.train()
    for ep in range(epochs):
        ep_loss = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            task_loss = loss_fn(clf(xb), yb)
            if use_replay:
                X_rep, y_rep = replay_buffer.sample()
                xr = torch.tensor(X_rep, dtype=torch.float32).to(device)
                yr = torch.tensor(y_rep, dtype=torch.long).to(device)
                replay_loss = F.cross_entropy(clf(xr), yr)
                loss = (1 - replay_lambda) * task_loss + replay_lambda * replay_loss
            else:
                loss = task_loss
            loss.backward()
            opt.step()
            ep_loss += loss.item()

        val_f1 = compute_macro_f1(clf, X_v, y_v, device)

        if not np.isnan(val_f1) and val_f1 - best_val_f1 >= es_min_delta:
            best_val_f1    = val_f1
            best_weights   = copy.deepcopy(clf.state_dict())
            patience_count = 0
        else:
            patience_count += 1

        if (ep + 1) % 10 == 0:
            print(f'  Epoch {ep+1:3d}/{epochs}  loss={ep_loss/max(len(dl),1):.4f}  val_macro_f1={val_f1:.3f}  patience={patience_count}/{es_patience}')

        if patience_count >= es_patience:
            print(f'  [Early Stop] Stopped at epoch {ep+1}. Best val_macro_f1={best_val_f1:.4f}')
            break

        clf.train()

    clf.load_state_dict(best_weights)
    clf.eval()
    return clf


print('Classifier defined (macro-F1 early stopping | weighted CE | replay buffer)')

Classifier defined (macro-F1 early stopping | weighted CE | replay buffer)


## Step 6 — ARC: Out-of-Task Detection (OTD)

In [67]:
def compute_batch_theta(logits_all, task_id, n_classes_per_task, theta):
    s             = n_classes_per_task
    past_boundary = s * task_id
    if past_boundary == 0:
        return theta
    with torch.no_grad():
        probs     = F.softmax(logits_all.float(), dim=-1)
        conf_all  = probs.max(dim=-1).values
        c_hat_all = probs[:, :past_boundary].max(dim=-1).values
        w_all     = conf_all / (c_hat_all + 1e-8)
    adaptive = float(w_all.median()) * 1.2
    # FIX: max se nahi, clip karo reasonable range mein
    result = float(np.clip(adaptive, theta, 3.0))  # 3.0 upper cap
    return result


def out_of_task_detection(logits, task_id, n_classes_per_task, epsilon, theta,
                           batch_theta=None):
    """
    OTD: Algorithm 1 from the ARC paper.

    Assumption 1 (RETENTION):
      pred in PAST-task class  AND  confidence >= epsilon
      → model strongly believes this is an old-task sample → retain.

    Assumption 2 (CORRECTION):
      pred in CURRENT-task class  AND  w = c / c_hat < theta
      where c     = max prob over ALL classes
            c_hat = max prob over PAST classes
      → current confidence is LOW relative to past → likely misclassified old sample.

    Paper Algorithm 1, line 9: if w < ϑ → correction  (w = c/c_hat)
    """
    s             = n_classes_per_task
    past_boundary = s * task_id

    probs    = F.softmax(logits.float(), dim=-1)
    pred_cls = probs.argmax(dim=-1)
    conf_all = probs.max(dim=-1).values

    effective_theta = batch_theta if batch_theta is not None else theta

    decisions = []
    for i in range(logits.shape[0]):
        pred_i = pred_cls[i].item()
        c      = conf_all[i].item()

        if past_boundary == 0:
            decisions.append('current')
            continue

        if pred_i < past_boundary:
            # Assumption 1: past-task prediction with high confidence → retention
            if c >= epsilon:
                decisions.append('retention')
            else:
                decisions.append('current')
        else:
            # Assumption 2: current-task pred, w = c/c_hat < theta → correction
            c_hat = probs[i, :past_boundary].max().item()
            w     = c / (c_hat + 1e-8)   # FIXED: paper says w = c/c_hat
            if w < effective_theta:        # FIXED: paper says w < ϑ (not >=)
                decisions.append('correction')
            else:
                decisions.append('current')

    return decisions, probs, pred_cls


print('OTD defined (paper-correct: w=c/c_hat, w<theta, adaptive=max)')

OTD defined (paper-correct: w=c/c_hat, w<theta, adaptive=max)


## Step 7 — Adaptive Retention

In [68]:
def adaptive_retention_step(clf, x_single, pseudo_label, lr, device):
    """
    One gradient update: L_CE(pseudo_label) + L_EM  (Eq. 2 & 3, ARC paper).
    SGD as per paper: 'one gradient update on each sample'.
    """
    clf.train()
    params = list(clf.parameters())[-2:]
    opt = torch.optim.SGD(clf.parameters(), lr=lr)
    opt.zero_grad()

    x = x_single.detach().clone().to(device)

    logits  = clf(x)
    probs   = F.softmax(logits, dim=-1)
    label_t = torch.tensor([pseudo_label], dtype=torch.long).to(device)

    L_CE = F.cross_entropy(logits, label_t)
    L_EM = -(probs * probs.clamp(min=1e-8).log()).sum(dim=-1).mean()

    loss = L_CE + L_EM
    loss.backward()
    opt.step()
    clf.eval()


print('Adaptive Retention defined (L_CE + L_EM, one SGD step per sample)')

Adaptive Retention defined (L_CE + L_EM, one SGD step per sample)


## Step 8 — Adaptive Correction (TSS)

In [21]:
def compute_tss(logits_single, task_id, n_classes_per_task, temperature):
    """
    Task-based Softmax Score (TSS) — Definition 1 from ARC paper.

    S_i = max_{s(i-1) <= k < s*i}  exp(z_k / T^(t-i))
           / sum_{j=0}^{s*i - 1}   exp(z_j / T^(t-i))

    Denominator = prefix sum up to task i (NOT all classes).
    Per-task temperature exponent T^(t-i).
    """
    s = n_classes_per_task
    t = task_id + 1
    z = logits_single.float()

    best_score = -float('inf')
    best_task  = 0
    best_cls   = 0

    for i in range(1, t + 1):
        exponent   = t - i
        temp_i     = temperature ** exponent

        task_logits  = z[s * (i - 1) : s * i]
        scaled_task  = task_logits / temp_i
        numerator    = scaled_task.exp().max()

        prefix_logits = z[: s * i]
        denominator   = (prefix_logits / temp_i).exp().sum()

        score = (numerator / (denominator + 1e-12)).item()

        if score > best_score:
            best_score = score
            best_task  = i - 1
            best_cls   = s * (i - 1) + task_logits.argmax().item()

    return best_task, best_cls


print('Adaptive Correction (TSS) defined')

Adaptive Correction (TSS) defined


## Step 8b — Hyperparameter Sweep for ε and θ (FIX-7/12/20)

In [ ]:
def sweep_arc_thresholds(tasks, X_tr, y_tr, X_val, y_val, feat_dim, cfg, device,
                          eps_values=None, theta_values=None):
    """
    FIX-20: clf_arc is now reset (deep-copied from checkpoint) before each
    (epsilon, theta) combination. Previously the same mutated clf_arc was
    reused across all combos, contaminating the sweep with prior retention
    updates and making all combos look artificially similar.
    """
    if eps_values is None:
        eps_values   = [0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
    if theta_values is None:
        theta_values = [0.02, 0.05, 0.08, 0.10, 0.15, 0.20]    

    # Train base classifier once through all tasks (no ARC, no replay)
    clf_base = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)
    clf_checkpoints_sweep = {}
    for task in tasks:
        tid = task['task_id']
        if tid > 0:
            clf_base.grow(len(task['new_classes']), device)
        clf_base = train_classifier_on_task(
            clf_base, X_tr, y_tr, X_val, y_val,
            task['all_classes'], cfg['clf_epochs'], cfg['clf_lr'],
            cfg['clf_batch'], device,
            es_patience=cfg.get('es_patience', 10),
            es_min_delta=cfg.get('es_min_delta', 1e-4),
        )
        clf_checkpoints_sweep[tid] = copy.deepcopy(clf_base.state_dict())

    print(f'\nSweeping ε ∈ {eps_values}  ×  θ ∈ {theta_values}')
    rows = []
    best_f1, best_eps, best_theta = -1.0, eps_values[0], theta_values[0]

    for eps in eps_values:
        for theta in theta_values:
            cfg_tmp = dict(cfg, arc_epsilon=eps, arc_theta=theta)

            task_f1s = []
            for task in tasks:
                tid      = task['task_id']
                seen_cls = task['all_classes']
                if tid == 0:
                    continue  # no OTD at task 0

                val_mask = np.isin(y_val, seen_cls)
                X_v, y_v = X_val[val_mask], y_val[val_mask]
                if len(X_v) == 0:
                    continue

                # FIX-20: Fresh clf_arc from checkpoint for EACH (eps, theta) combo
                clf_arc = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)
                for prev_task in tasks[:tid + 1]:
                    if prev_task['task_id'] > 0:
                        clf_arc.grow(len(prev_task['new_classes']), device)
                clf_arc.load_state_dict(clf_checkpoints_sweep[tid])
                clf_arc.eval()

                Xv = torch.tensor(X_v, dtype=torch.float32).to(device)
                with torch.no_grad():
                    logits_all = clf_arc(Xv)
                batch_theta_val = compute_batch_theta(
                    logits_all, tid, cfg['classes_per_task'], theta)

                val_preds = []
                for i in range(len(Xv)):
                    xi = Xv[i:i+1]
                    with torch.no_grad():
                        li = clf_arc(xi)
                    decisions, _, pred_i = out_of_task_detection(
                        li, tid, cfg['classes_per_task'], eps, theta,
                        batch_theta=batch_theta_val)
                    decision     = decisions[0]
                    pseudo_label = pred_i[0].item()

                    if decision == 'retention':
                        adaptive_retention_step(clf_arc, xi, pseudo_label,
                                                cfg_tmp['arc_lr'], device)
                        with torch.no_grad():
                            clf_arc.eval()
                            li = clf_arc(xi)
                        pred_final = li.argmax(1).item()
                    elif decision == 'correction':
                        with torch.no_grad():
                            clf_arc.eval()
                            li = clf_arc(xi)
                        _, pred_final = compute_tss(li[0], tid, cfg['classes_per_task'],
                                                    cfg_tmp['arc_temp'])
                    else:
                        pred_final = pseudo_label

                    val_preds.append(pred_final)

                present  = np.unique(y_v)
                task_f1  = float(f1_score(y_v, val_preds, labels=present,
                                          average='macro', zero_division=0))
                task_f1s.append(task_f1)

            if not task_f1s:
                continue
            macro_f1 = float(np.mean(task_f1s))
            rows.append({'epsilon': eps, 'theta': theta, 'val_macro_f1': macro_f1})
            if macro_f1 > best_f1:
                best_f1, best_eps, best_theta = macro_f1, eps, theta

    sweep_df = pd.DataFrame(rows).sort_values('val_macro_f1', ascending=False)
    print('\nTop-5 threshold combinations (avg over all tasks):')
    print(sweep_df.head(5).to_string(index=False))
    print(f'\nBest: ε={best_eps}  θ={best_theta}  avg_val_macro_f1={best_f1:.4f}')
    return best_eps, best_theta, sweep_df


print('Hyperparameter sweep defined (FIX-20: reset clf_arc for each combo)')

Hyperparameter sweep defined (FIX-20: reset clf_arc for each combo)


In [65]:
# Sweep cell mein ya manually:
atc_best_eps = 0.95   # 0.8 → 0.95 — sirf very high confidence pe retain karo
atc_best_theta = 0.10  # 0.10 — sirf very high confidence pe retain karo
CFG_ATC_ARC = dict(CFG, arc_epsilon=atc_best_eps, arc_theta=atc_best_theta)

## Step 9 — Evaluation: Accuracy + Macro-F1 + AUROC + Forgetting

In [24]:
def compute_auroc(y_t, probs_np, seen_cls, n_total_cls):
    valid_seen = [c for c in seen_cls if c < n_total_cls]
    if len(valid_seen) < 2:
        return float('nan')
    y_t_arr    = np.asarray(y_t)
    y_bin      = np.stack([(y_t_arr == c).astype(int) for c in valid_seen], axis=1)
    probs_seen = probs_np[:, valid_seen]
    probs_seen = probs_seen / (probs_seen.sum(axis=1, keepdims=True) + 1e-8)
    auroc_list = []
    for col_i in range(len(valid_seen)):
        col_true = y_bin[:, col_i]
        col_prob = probs_seen[:, col_i]
        if col_true.sum() == 0 or col_true.sum() == len(col_true):
            continue
        try:
            auroc_list.append(roc_auc_score(col_true, col_prob))
        except Exception:
            pass
    return float(np.mean(auroc_list)) if auroc_list else float('nan')


def evaluate_task(
    clf, X_test, y_test, seen_cls,
    task_id=None, n_cpt=None, epsilon=None, theta=None, temp=None,
    arc_lr=None, use_arc=False, arc_mode='full', device='cpu',
    persistent_clf_arc=None,
):
    mask = np.isin(y_test, seen_cls)
    X_t, y_t = X_test[mask], y_test[mask]
    if len(X_t) == 0:
        return {'accuracy': 0.0, 'macro_f1': 0.0, 'auroc': float('nan')}, persistent_clf_arc

    Xt = torch.tensor(X_t, dtype=torch.float32).to(device)
    clf.eval()
    with torch.no_grad():
        logits_all = clf(Xt)
    n_total_cls = logits_all.shape[1]

    if use_arc and task_id is not None and task_id > 0:
        # FIX-C: use persistent_clf_arc as-is (carries state across test stream)
        clf_arc = persistent_clf_arc if persistent_clf_arc is not None else copy.deepcopy(clf)

        with torch.no_grad():
            clf_arc.eval()
            logits_for_theta = clf_arc(Xt)
        batch_theta = compute_batch_theta(logits_for_theta, task_id, n_cpt, theta)

        otd_counts = {'retention': 0, 'correction': 0, 'current': 0}
        preds, adapted_logits_list = [], []

        for i in range(len(Xt)):
            xi = Xt[i:i+1]

            with torch.no_grad():
                clf_arc.eval()
                li_otd = clf_arc(xi)

            decisions, _, pred_i = out_of_task_detection(
                li_otd, task_id, n_cpt, epsilon, theta,
                batch_theta=batch_theta,
            )
            decision     = decisions[0]
            pseudo_label = pred_i[0].item()
            otd_counts[decision] += 1

            if decision == 'retention' and arc_mode in ('full', 'retention'):
                adaptive_retention_step(clf_arc, xi, pseudo_label, arc_lr, device)
                with torch.no_grad():
                    clf_arc.eval()
                    li_final = clf_arc(xi)
                pred_final = li_final.argmax(1).item()

            elif decision == 'correction' and arc_mode in ('full', 'correction'):
                with torch.no_grad():
                    clf_arc.eval()
                    li_final = clf_arc(xi)
                _, pred_final = compute_tss(li_final[0], task_id, n_cpt, temp)

            else:
                li_final   = li_otd
                pred_final = pseudo_label

            preds.append(pred_final)
            adapted_logits_list.append(li_final.detach())

        preds = np.array(preds)
        adapted_logits = torch.cat(adapted_logits_list, dim=0)
        with torch.no_grad():
            probs_np = F.softmax(adapted_logits, dim=-1).cpu().numpy()

        n = len(Xt)
        print(f'    OTD [mode={arc_mode}]: retention={otd_counts["retention"]}/{n} '
              f'correction={otd_counts["correction"]}/{n} '
              f'current={otd_counts["current"]}/{n}  '
              f'(batch_theta={batch_theta:.3f})')

    else:
        clf_arc = persistent_clf_arc  # unchanged
        with torch.no_grad():
            preds    = logits_all.argmax(1).cpu().numpy()
            probs_np = F.softmax(logits_all, dim=-1).cpu().numpy()

    accuracy = float((preds == y_t).mean())
    present  = np.unique(y_t)
    macro_f1 = float(f1_score(y_t, preds, labels=present, average='macro', zero_division=0))
    auroc    = compute_auroc(y_t, probs_np, seen_cls, n_total_cls)

    # FIX-C: return updated clf_arc so pipeline can carry state forward
    return {'accuracy': accuracy, 'macro_f1': macro_f1, 'auroc': auroc}, clf_arc


print('evaluate_task defined (FIX-C: returns updated clf_arc)')

evaluate_task defined (FIX-C: returns updated clf_arc)


## Step 10 — Full CIL Pipeline

In [25]:
def run_cil_pipeline(
    tasks, X_tr, y_tr, X_val, y_val, X_te, y_te,
    feat_dim, label_map, cfg, device, use_arc=False, arc_mode='full', name='Dataset'
):
    """
    FIX-19: evaluate_task now always receives task_id=tid (current training task).
    Previously task_id=prev_tid caused OTD boundary=0 when evaluating old tasks,
    making ARC a no-op for all prior tasks — the primary cause of ARC under-performance.
    """
    print(f'\n{"="*60}')
    print(f'  {name} CIL Pipeline  |  ARC={use_arc}  arc_mode={arc_mode}')
    print(f'{"="*60}')

    clf     = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)
    buf     = ReplayBuffer(max_per_class=cfg.get('replay_buf_size', 20))
    buf_rng = np.random.default_rng(cfg['seed'])
    R       = defaultdict(dict)
    peak_acc = {}
    rows     = []

    use_replay = not use_arc
    if use_arc:
        print(f'  [Replay OFF] — ARC run is memory-free (paper Section 3)')
    else:
        print(f'  [Replay ON]  — Baseline uses experience replay')

    persistent_clf_arc = None
    clf_checkpoints = {}

    for task in tasks:
        tid      = task['task_id']
        seen_cls = task['all_classes']

        print(f'\n-- Task {tid} | Classes: {list(task["class_names"].values())} --')

        if tid > 0:
            clf.grow(len(task['new_classes']), device)

        clf = train_classifier_on_task(
            clf, X_tr, y_tr, X_val, y_val,
            seen_cls, cfg['clf_epochs'], cfg['clf_lr'], cfg['clf_batch'], device,
            replay_buffer = buf if use_replay else None,
            replay_lambda = cfg.get('replay_lambda', 0.5),
            es_patience   = cfg.get('es_patience', 10),
            es_min_delta  = cfg.get('es_min_delta', 1e-4),
        )

        if use_replay:
            tr_mask = np.isin(y_tr, seen_cls)
            buf.update(X_tr[tr_mask], y_tr[tr_mask], rng=buf_rng)

        clf_checkpoints[tid] = copy.deepcopy(clf.state_dict())

        if use_arc:
            persistent_clf_arc = copy.deepcopy(clf)

        for prev_task in tasks[:tid + 1]:
            prev_tid  = prev_task['task_id']
            prev_seen = prev_task['all_classes']

            res,persistent_clf_arc = evaluate_task(
                clf, X_te, y_te, prev_seen,
                task_id            = tid,       # FIX-19: always current task, not prev_tid
                n_cpt              = cfg['classes_per_task'],
                epsilon            = cfg['arc_epsilon'],
                theta              = cfg['arc_theta'],
                temp               = cfg['arc_temp'],
                arc_lr             = cfg['arc_lr'],
                use_arc            = use_arc,
                arc_mode           = arc_mode,
                device             = device,
                persistent_clf_arc = persistent_clf_arc,
            )
            R[tid][prev_tid] = res

            if prev_tid == tid:
                peak_acc[tid] = res['accuracy']

            auroc_str = f"{res['auroc']:.4f}" if not np.isnan(res['auroc']) else 'nan'
            print(f"  Eval Task {prev_tid} → acc={res['accuracy']:.4f}  macro_f1={res['macro_f1']:.4f}  auroc={auroc_str}")

    n_tasks   = len(tasks)
    final_tid = n_tasks - 1
    acc_list, f1_list, auroc_list, forget_list = [], [], [], []

    for prev_tid in range(n_tasks):
        final_acc    = R[final_tid][prev_tid]['accuracy']
        final_f1     = R[final_tid][prev_tid]['macro_f1']
        final_auroc  = R[final_tid][prev_tid]['auroc']
        acc_list.append(final_acc)
        f1_list.append(final_f1)
        auroc_list.append(final_auroc)
        if prev_tid < final_tid:
            forget_list.append(peak_acc[prev_tid] - final_acc)

        rows.append({
            'task_id'       : prev_tid,
            'class_names'   : str(list(tasks[prev_tid]['class_names'].values())),
            'final_acc'     : round(final_acc, 4),
            'final_macro_f1': round(final_f1, 4),
            'final_auroc'   : round(final_auroc, 4) if not np.isnan(final_auroc) else float('nan'),
            'peak_acc'      : round(peak_acc.get(prev_tid, final_acc), 4),
            'forgetting'    : round(peak_acc.get(prev_tid, final_acc) - final_acc, 4),
        })

    avg_acc    = float(np.mean(acc_list))
    avg_f1     = float(np.mean(f1_list))
    forgetting = float(np.mean(forget_list)) if forget_list else 0.0
    avg_auroc  = float(np.nanmean(auroc_list))

    print(f'\n{"─"*50}')
    print(f'  Average Accuracy (AB)   : {avg_acc:.4f}')
    print(f'  Average Macro-F1        : {avg_f1:.4f}')
    print(f'  Forgetting (F)          : {forgetting:.4f}')
    print(f'  Average AUROC           : {avg_auroc:.4f}')
    print(f'{"─"*50}')

    return pd.DataFrame(rows), avg_acc, avg_f1, forgetting, avg_auroc, clf_checkpoints


print('CIL pipeline defined (FIX-19: task_id=tid for correct OTD boundary)')

CIL pipeline defined (FIX-19: task_id=tid for correct OTD boundary)


## Step 10b — No-Replay Baseline

In [40]:
def run_cil_pipeline_no_replay(
    tasks, X_tr, y_tr, X_val, y_val, X_te, y_te,
    feat_dim, label_map, cfg, device, name='Dataset'
):
    print(f'\n{"="*60}')
    print(f'  {name} CIL Pipeline  |  ARC=False  Replay=OFF (no-replay baseline)')
    print(f'{"="*60}')

    clf     = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)
    R       = defaultdict(dict)
    peak_acc = {}
    rows     = []
    clf_checkpoints = {}

    for task in tasks:
        tid      = task['task_id']
        seen_cls = task['all_classes']

        print(f'\n-- Task {tid} | Classes: {list(task["class_names"].values())} --')

        if tid > 0:
            clf.grow(len(task['new_classes']), device)

        clf = train_classifier_on_task(
            clf, X_tr, y_tr, X_val, y_val,
            seen_cls, cfg['clf_epochs'], cfg['clf_lr'], cfg['clf_batch'], device,
            replay_buffer = None,
            es_patience   = cfg.get('es_patience', 10),
            es_min_delta  = cfg.get('es_min_delta', 1e-4),
        )

        clf_checkpoints[tid] = copy.deepcopy(clf.state_dict())

        for prev_task in tasks[:tid + 1]:
            prev_tid  = prev_task['task_id']
            prev_seen = prev_task['all_classes']

            res, _ = evaluate_task(          # ← tuple unpack
                clf, X_te, y_te, prev_seen,
                task_id=tid, n_cpt=cfg['classes_per_task'],
                epsilon=cfg['arc_epsilon'], theta=cfg['arc_theta'],
                temp=cfg['arc_temp'], arc_lr=cfg['arc_lr'],
                use_arc=False, device=device,
            )
            R[tid][prev_tid] = res

            if prev_tid == tid:
                peak_acc[tid] = res['accuracy']

            auroc_str = f"{res['auroc']:.4f}" if not np.isnan(res['auroc']) else 'nan'
            print(f"  Eval Task {prev_tid} → acc={res['accuracy']:.4f}  macro_f1={res['macro_f1']:.4f}  auroc={auroc_str}")

    n_tasks   = len(tasks)
    final_tid = n_tasks - 1
    acc_list, f1_list, auroc_list, forget_list = [], [], [], []

    for prev_tid in range(n_tasks):
        final_acc   = R[final_tid][prev_tid]['accuracy']
        final_f1    = R[final_tid][prev_tid]['macro_f1']
        final_auroc = R[final_tid][prev_tid]['auroc']
        acc_list.append(final_acc)
        f1_list.append(final_f1)
        auroc_list.append(final_auroc)
        if prev_tid < final_tid:
            forget_list.append(peak_acc[prev_tid] - final_acc)

        rows.append({
            'task_id'       : prev_tid,
            'class_names'   : str(list(tasks[prev_tid]['class_names'].values())),
            'final_acc'     : round(final_acc, 4),
            'final_macro_f1': round(final_f1, 4),
            'final_auroc'   : round(final_auroc, 4) if not np.isnan(final_auroc) else float('nan'),
            'peak_acc'      : round(peak_acc.get(prev_tid, final_acc), 4),
            'forgetting'    : round(peak_acc.get(prev_tid, final_acc) - final_acc, 4),
        })

    avg_acc    = float(np.mean(acc_list))
    avg_f1     = float(np.mean(f1_list))
    forgetting = float(np.mean(forget_list)) if forget_list else 0.0
    avg_auroc  = float(np.nanmean(auroc_list))

    print(f'\n{"─"*50}')
    print(f'  Average Accuracy (AB)   : {avg_acc:.4f}')
    print(f'  Average Macro-F1        : {avg_f1:.4f}')
    print(f'  Forgetting (F)          : {forgetting:.4f}')
    print(f'  Average AUROC           : {avg_auroc:.4f}')
    print(f'{"─"*50}')

    return pd.DataFrame(rows), avg_acc, avg_f1, forgetting, avg_auroc, clf_checkpoints


print('No-replay baseline pipeline defined')

No-replay baseline pipeline defined


## Step 10c — Ablation Study: Retention-only vs Correction-only vs Full ARC

Matches Table 4 in the paper — isolates the contribution of each ARC component.

In [41]:
def run_ablation_study(
    tasks, X_tr, y_tr, X_val, y_val, X_te, y_te,
    feat_dim, label_map, cfg, device, dataset_name='Dataset'
):
    """
    Ablation study matching Table 4 from the ARC paper.

    Runs four conditions on the SAME base classifier (trained once, no replay):
      1. Baseline       : no ARC at all
      2. Retention-only : only Adaptive Retention fires (arc_mode='retention')
      3. Correction-only: only Adaptive Correction fires (arc_mode='correction')
      4. Full ARC       : both components active (arc_mode='full')

    All four share the same trained clf weights (from no-replay training),
    so any differences are purely due to the ARC test-time mechanism.
    """
    print(f'\n{"="*60}')
    print(f'  ABLATION STUDY — {dataset_name}')
    print(f'{"="*60}')

    # ── Train base classifier (no ARC, no replay) ──────────────────────────
    clf_base = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)
    clf_checkpoints = {}

    print('\nTraining base classifier (shared across all ablation conditions)...')
    for task in tasks:
        tid      = task['task_id']
        seen_cls = task['all_classes']
        print(f'  Task {tid}: {list(task["class_names"].values())}')
        if tid > 0:
            clf_base.grow(len(task['new_classes']), device)
        clf_base = train_classifier_on_task(
            clf_base, X_tr, y_tr, X_val, y_val,
            seen_cls, cfg['clf_epochs'], cfg['clf_lr'], cfg['clf_batch'], device,
            es_patience=cfg.get('es_patience', 10),
            es_min_delta=cfg.get('es_min_delta', 1e-4),
        )
        clf_checkpoints[tid] = copy.deepcopy(clf_base.state_dict())

    # ── Evaluate all four conditions ────────────────────────────────────────
    conditions = [
        ('Baseline',        False, 'full'),
        ('Retention-only',  True,  'retention'),
        ('Correction-only', True,  'correction'),
        ('Full ARC',        True,  'full'),
    ]

    ablation_rows = []

    for cond_name, use_arc, arc_mode in conditions:
        print(f'\n--- Condition: {cond_name} ---')
        n_tasks   = len(tasks)
        final_tid = n_tasks - 1

        # Restore classifier to final-task checkpoint
        clf_eval = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)
        for task in tasks:
            if task['task_id'] > 0:
                clf_eval.grow(len(task['new_classes']), device)
        clf_eval.load_state_dict(clf_checkpoints[final_tid])
        clf_eval.eval()

        # For ARC conditions: persistent clf_arc carries retention updates across
        # prev_task evaluations in the final task loop.
        persistent_clf_arc = copy.deepcopy(clf_eval) if use_arc else None

        # Evaluate against all prior tasks at the final task boundary
        acc_list, f1_list, auroc_list, forget_list = [], [], [], []
        peak_accs = {}

        # Compute peak accuracy at each task's own training step
        for task in tasks:
            tid = task['task_id']
            clf_t = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)
            for t2 in tasks[:tid+1]:
                if t2['task_id'] > 0:
                    clf_t.grow(len(t2['new_classes']), device)
            clf_t.load_state_dict(clf_checkpoints[tid])
            clf_t.eval()
            res_peak = evaluate_task(
                clf_t, X_te, y_te, task['all_classes'],
                task_id=tid, n_cpt=cfg['classes_per_task'],
                epsilon=cfg['arc_epsilon'], theta=cfg['arc_theta'],
                temp=cfg['arc_temp'], arc_lr=cfg['arc_lr'],
                use_arc=False, device=device,  # peak always measured without ARC
            )
            peak_accs[tid] = res_peak['accuracy']

        # Evaluate all tasks at final task boundary
        for prev_task in tasks:
            prev_tid  = prev_task['task_id']
            prev_seen = prev_task['all_classes']

            res = evaluate_task(
                clf_eval, X_te, y_te, prev_seen,
                task_id            = final_tid,  # FIX-19: always final task
                n_cpt              = cfg['classes_per_task'],
                epsilon            = cfg['arc_epsilon'],
                theta              = cfg['arc_theta'],
                temp               = cfg['arc_temp'],
                arc_lr             = cfg['arc_lr'],
                use_arc            = use_arc,
                arc_mode           = arc_mode,
                device             = device,
                persistent_clf_arc = persistent_clf_arc,
            )
            acc_list.append(res['accuracy'])
            f1_list.append(res['macro_f1'])
            auroc_list.append(res['auroc'])
            if prev_tid < final_tid:
                forget_list.append(peak_accs[prev_tid] - res['accuracy'])

            print(f"  Task {prev_tid}: acc={res['accuracy']:.4f}  f1={res['macro_f1']:.4f}")

        avg_acc    = float(np.mean(acc_list))
        avg_f1     = float(np.mean(f1_list))
        forgetting = float(np.mean(forget_list)) if forget_list else 0.0
        avg_auroc  = float(np.nanmean(auroc_list))

        print(f'  → AB={avg_acc:.4f}  Macro-F1={avg_f1:.4f}  F={forgetting:.4f}  AUROC={avg_auroc:.4f}')

        ablation_rows.append({
            'Dataset'  : dataset_name,
            'Condition': cond_name,
            'AB'       : round(avg_acc,    4),
            'Macro-F1' : round(avg_f1,     4),
            'F'        : round(forgetting, 4),
            'AUROC'    : round(avg_auroc,  4),
        })

    ablation_df = pd.DataFrame(ablation_rows)

    # Compute incremental gains (matching Table 4 format)
    baseline_ab = ablation_df.loc[ablation_df['Condition'] == 'Baseline', 'AB'].values[0]
    baseline_f1 = ablation_df.loc[ablation_df['Condition'] == 'Baseline', 'Macro-F1'].values[0]
    ablation_df['ΔAB']      = (ablation_df['AB']       - baseline_ab).round(4)
    ablation_df['ΔMacro-F1'] = (ablation_df['Macro-F1'] - baseline_f1).round(4)

    print(f'\n{"─"*60}')
    print(f'  ABLATION SUMMARY — {dataset_name}')
    print(f'{"─"*60}')
    print(ablation_df.to_string(index=False))

    return ablation_df


print('Ablation study defined (Table 4 equivalent: Retention-only | Correction-only | Full ARC)')

Ablation study defined (Table 4 equivalent: Retention-only | Correction-only | Full ARC)


## Step 11 — BWT / FWT from Saved Checkpoints

In [42]:
def compute_bwt_fwt_from_checkpoints(
    tasks, X_te, y_te, feat_dim, clf_checkpoints, cfg, device, use_arc=False
):
    n_tasks = len(tasks)
    if n_tasks < 2:
        return float('nan'), float('nan')

    R = {}
    clf_tmp = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)

    for task in tasks:
        tid      = task['task_id']
        seen_cls = task['all_classes']

        if tid > 0:
            clf_tmp.grow(len(task['new_classes']), device)

        clf_tmp.load_state_dict(clf_checkpoints[tid])
        clf_tmp.eval()

        R[tid] = {}
        persistent_clf_arc_bwt = copy.deepcopy(clf_tmp) if use_arc else None

        for prev_task in tasks[:tid + 1]:
            prev_tid  = prev_task['task_id']
            prev_seen = prev_task['all_classes']
            res = evaluate_task(
                clf_tmp, X_te, y_te, prev_seen,
                task_id=tid, n_cpt=cfg['classes_per_task'],  # FIX-19: tid not prev_tid
                epsilon=cfg['arc_epsilon'], theta=cfg['arc_theta'],
                temp=cfg['arc_temp'], arc_lr=cfg['arc_lr'],
                use_arc=use_arc, device=device,
                persistent_clf_arc=persistent_clf_arc_bwt,
            )
            R[tid][prev_tid] = res['accuracy']

    final_tid = n_tasks - 1
    peak_acc_bwt = {i: R[i][i] for i in range(n_tasks)}
    bwt_vals = [R[final_tid][i] - peak_acc_bwt[i] for i in range(n_tasks - 1)]
    bwt = float(np.mean(bwt_vals))

    fwt_vals = []
    for i in range(1, n_tasks):
        n_cls_i  = tasks[i]['n_classes']
        chance_i = 1.0 / n_cls_i
        if i - 1 in R and i in R.get(i - 1, {}):
            fwt_vals.append(R[i - 1][i] - chance_i)
        else:
            fwt_vals.append(R[i][i] - chance_i)
    fwt = float(np.mean(fwt_vals)) if fwt_vals else float('nan')

    return bwt, fwt


print('BWT/FWT from checkpoints defined (FIX-9 + FIX-19)')

BWT/FWT from checkpoints defined (FIX-9 + FIX-19)


## Run Hyperparameter Sweep

In [43]:
print('=== ATC Threshold Sweep (v12) ===')
atc_best_eps, atc_best_theta, atc_sweep_df = sweep_arc_thresholds(
    atc_tasks, atc_X_tr, atc_y_tr, atc_X_val, atc_y_val,
    FEAT_DIM, CFG, DEVICE,
    eps_values   = [0.30, 0.40, 0.50, 0.60, 0.70, 0.80],
    theta_values = [0.02, 0.05, 0.08, 0.10, 0.15, 0.20]
)

print('\n=== DrugBank Threshold Sweep (v12) ===')
db_best_eps, db_best_theta, db_sweep_df = sweep_arc_thresholds(
    db_tasks, db_X_tr, db_y_tr, db_X_val, db_y_val,
    FEAT_DIM, CFG, DEVICE,
    eps_values   = [0.30, 0.40, 0.50, 0.60, 0.70, 0.80],
    theta_values = [0.02, 0.05, 0.08, 0.10, 0.15, 0.20]
)

=== ATC Threshold Sweep (v12) ===
  Epoch  10/100  loss=0.3395  val_macro_f1=0.427  patience=8/10
  [Early Stop] Stopped at epoch 12. Best val_macro_f1=0.4673
  Epoch  10/100  loss=0.8122  val_macro_f1=0.280  patience=4/10
  Epoch  20/100  loss=0.6545  val_macro_f1=0.303  patience=0/10
  Epoch  30/100  loss=0.5617  val_macro_f1=0.299  patience=9/10
  [Early Stop] Stopped at epoch 31. Best val_macro_f1=0.3157
  Epoch  10/100  loss=0.8298  val_macro_f1=0.325  patience=4/10
  Epoch  20/100  loss=0.7074  val_macro_f1=0.409  patience=6/10
  [Early Stop] Stopped at epoch 24. Best val_macro_f1=0.4088
  Epoch  10/100  loss=0.8584  val_macro_f1=0.380  patience=1/10
  Epoch  20/100  loss=0.7482  val_macro_f1=0.392  patience=3/10
  [Early Stop] Stopped at epoch 27. Best val_macro_f1=0.4009
  Epoch  10/100  loss=0.9095  val_macro_f1=0.375  patience=3/10
  [Early Stop] Stopped at epoch 17. Best val_macro_f1=0.3966
  Epoch  10/100  loss=1.0099  val_macro_f1=0.327  patience=9/10
  [Early Stop] Stoppe

## Run — ATC (All Conditions)

In [69]:
# Baseline with replay
atc_res_replay, atc_AB_replay, atc_F1_replay, atc_F_replay, atc_AUROC_replay, atc_ckpts_replay = run_cil_pipeline(
    tasks=atc_tasks, X_tr=atc_X_tr, y_tr=atc_y_tr,
    X_val=atc_X_val, y_val=atc_y_val, X_te=atc_X_te, y_te=atc_y_te,
    feat_dim=FEAT_DIM, label_map=atc_label_map,
    cfg=CFG, device=DEVICE, use_arc=False, name='ATC Replay-Baseline'
)
print('\nATC Replay-Baseline Results:')
print(atc_res_replay.to_string(index=False))


  ATC Replay-Baseline CIL Pipeline  |  ARC=False  arc_mode=full
  [Replay ON]  — Baseline uses experience replay

-- Task 0 | Classes: ['ATC-0', 'ATC-1'] --
  Epoch  10/100  loss=0.3433  val_macro_f1=0.409  patience=8/10
  [Early Stop] Stopped at epoch 12. Best val_macro_f1=0.4375
  Eval Task 0 → acc=0.7213  macro_f1=0.5724  auroc=0.7607

-- Task 1 | Classes: ['ATC-0', 'ATC-1', 'ATC-2', 'ATC-3'] --
  Epoch  10/100  loss=0.4935  val_macro_f1=0.238  patience=4/10
  Epoch  20/100  loss=0.3923  val_macro_f1=0.302  patience=2/10
  Epoch  30/100  loss=0.3368  val_macro_f1=0.316  patience=1/10
  Epoch  40/100  loss=0.2977  val_macro_f1=0.340  patience=0/10
  Epoch  50/100  loss=0.2692  val_macro_f1=0.301  patience=10/10
  [Early Stop] Stopped at epoch 50. Best val_macro_f1=0.3400
  Eval Task 0 → acc=0.4918  macro_f1=0.4762  auroc=0.7906
  Eval Task 1 → acc=0.4626  macro_f1=0.4046  auroc=0.7074

-- Task 2 | Classes: ['ATC-0', 'ATC-1', 'ATC-2', 'ATC-3', 'ATC-4', 'ATC-5'] --
  Epoch  10/100  lo

In [70]:
# Memory-free baseline
atc_res_noreplay, atc_AB_norepay, atc_F1_norepay, atc_F_norepay, atc_AUROC_norepay, atc_ckpts_norepay = run_cil_pipeline_no_replay(
    tasks=atc_tasks, X_tr=atc_X_tr, y_tr=atc_y_tr,
    X_val=atc_X_val, y_val=atc_y_val, X_te=atc_X_te, y_te=atc_y_te,
    feat_dim=FEAT_DIM, label_map=atc_label_map,
    cfg=CFG, device=DEVICE, name='ATC No-Replay-Baseline'
)
print('\nATC No-Replay-Baseline Results:')
print(atc_res_noreplay.to_string(index=False))


  ATC No-Replay-Baseline CIL Pipeline  |  ARC=False  Replay=OFF (no-replay baseline)

-- Task 0 | Classes: ['ATC-0', 'ATC-1'] --
  Epoch  10/100  loss=0.3413  val_macro_f1=0.448  patience=0/10
  Epoch  20/100  loss=0.2476  val_macro_f1=0.438  patience=10/10
  [Early Stop] Stopped at epoch 20. Best val_macro_f1=0.4483
  Eval Task 0 → acc=0.7541  macro_f1=0.5982  auroc=0.7735

-- Task 1 | Classes: ['ATC-0', 'ATC-1', 'ATC-2', 'ATC-3'] --
  Epoch  10/100  loss=0.7779  val_macro_f1=0.249  patience=7/10
  Epoch  20/100  loss=0.6334  val_macro_f1=0.279  patience=2/10
  [Early Stop] Stopped at epoch 28. Best val_macro_f1=0.2970
  Eval Task 0 → acc=0.4426  macro_f1=0.4467  auroc=0.7650
  Eval Task 1 → acc=0.5034  macro_f1=0.4527  auroc=0.7238

-- Task 2 | Classes: ['ATC-0', 'ATC-1', 'ATC-2', 'ATC-3', 'ATC-4', 'ATC-5'] --
  Epoch  10/100  loss=0.8524  val_macro_f1=0.351  patience=0/10
  Epoch  20/100  loss=0.7001  val_macro_f1=0.344  patience=1/10
  Epoch  30/100  loss=0.6124  val_macro_f1=0.39

In [71]:
# ARC with tuned thresholds
CFG_ATC_ARC = dict(CFG, arc_epsilon=atc_best_eps, arc_theta=atc_best_theta)
print(f'ATC ARC thresholds: ε={atc_best_eps}  θ={atc_best_theta}')

atc_res_arc, atc_AB_arc, atc_F1_arc, atc_F_arc, atc_AUROC_arc, atc_ckpts_arc = run_cil_pipeline(
    tasks=atc_tasks, X_tr=atc_X_tr, y_tr=atc_y_tr,
    X_val=atc_X_val, y_val=atc_y_val, X_te=atc_X_te, y_te=atc_y_te,
    feat_dim=FEAT_DIM, label_map=atc_label_map,
    cfg=CFG_ATC_ARC, device=DEVICE, use_arc=True, name='ATC + ARC'
)
print('\nATC + ARC Results:')
print(atc_res_arc.to_string(index=False))

ATC ARC thresholds: ε=0.95  θ=0.1

  ATC + ARC CIL Pipeline  |  ARC=True  arc_mode=full
  [Replay OFF] — ARC run is memory-free (paper Section 3)

-- Task 0 | Classes: ['ATC-0', 'ATC-1'] --
  Epoch  10/100  loss=0.3475  val_macro_f1=0.360  patience=2/10
  Epoch  20/100  loss=0.2490  val_macro_f1=0.416  patience=1/10
  Epoch  30/100  loss=0.1969  val_macro_f1=0.470  patience=2/10
  [Early Stop] Stopped at epoch 38. Best val_macro_f1=0.4699
  Eval Task 0 → acc=0.8033  macro_f1=0.6090  auroc=0.7692

-- Task 1 | Classes: ['ATC-0', 'ATC-1', 'ATC-2', 'ATC-3'] --
  Epoch  10/100  loss=0.7197  val_macro_f1=0.286  patience=3/10
  Epoch  20/100  loss=0.6028  val_macro_f1=0.315  patience=0/10
  Epoch  30/100  loss=0.5371  val_macro_f1=0.307  patience=2/10
  Epoch  40/100  loss=0.4981  val_macro_f1=0.297  patience=2/10
  [Early Stop] Stopped at epoch 48. Best val_macro_f1=0.3440
    OTD [mode=full]: retention=5/61 correction=3/61 current=53/61  (batch_theta=1.200)
  Eval Task 0 → acc=0.5738  macro

## Run — DrugBank (All Conditions)

In [48]:
db_res_replay, db_AB_replay, db_F1_replay, db_F_replay, db_AUROC_replay, db_ckpts_replay = run_cil_pipeline(
    tasks=db_tasks, X_tr=db_X_tr, y_tr=db_y_tr,
    X_val=db_X_val, y_val=db_y_val, X_te=db_X_te, y_te=db_y_te,
    feat_dim=FEAT_DIM, label_map=db_label_map,
    cfg=CFG, device=DEVICE, use_arc=False, name='DrugBank Replay-Baseline'
)

db_res_norepay, db_AB_norepay, db_F1_norepay, db_F_norepay, db_AUROC_norepay, db_ckpts_norepay = run_cil_pipeline_no_replay(
    tasks=db_tasks, X_tr=db_X_tr, y_tr=db_y_tr,
    X_val=db_X_val, y_val=db_y_val, X_te=db_X_te, y_te=db_y_te,
    feat_dim=FEAT_DIM, label_map=db_label_map,
    cfg=CFG, device=DEVICE, name='DrugBank No-Replay-Baseline'
)

CFG_DB_ARC = dict(CFG, arc_epsilon=db_best_eps, arc_theta=db_best_theta)
print(f'DrugBank ARC thresholds: ε={db_best_eps}  θ={db_best_theta}')

db_res_arc, db_AB_arc, db_F1_arc, db_F_arc, db_AUROC_arc, db_ckpts_arc = run_cil_pipeline(
    tasks=db_tasks, X_tr=db_X_tr, y_tr=db_y_tr,
    X_val=db_X_val, y_val=db_y_val, X_te=db_X_te, y_te=db_y_te,
    feat_dim=FEAT_DIM, label_map=db_label_map,
    cfg=CFG_DB_ARC, device=DEVICE, use_arc=True, name='DrugBank + ARC'
)


  DrugBank Replay-Baseline CIL Pipeline  |  ARC=False  arc_mode=full
  [Replay ON]  — Baseline uses experience replay

-- Task 0 | Classes: ['carbonyl', 'nitro'] --
  Epoch  10/100  loss=0.0257  val_macro_f1=0.855  patience=9/10
  [Early Stop] Stopped at epoch 11. Best val_macro_f1=0.8623
  Eval Task 0 → acc=0.9930  macro_f1=0.6649  auroc=1.0000

-- Task 1 | Classes: ['carbonyl', 'nitro', 'none_detected', 'sulfonyl'] --
  Epoch  10/100  loss=0.0736  val_macro_f1=0.810  patience=3/10
  Epoch  20/100  loss=0.0492  val_macro_f1=0.839  patience=2/10
  Epoch  30/100  loss=0.0391  val_macro_f1=0.884  patience=5/10
  [Early Stop] Stopped at epoch 35. Best val_macro_f1=0.8940
  Eval Task 0 → acc=0.9235  macro_f1=0.6229  auroc=1.0000
  Eval Task 1 → acc=0.9269  macro_f1=0.7163  auroc=0.9909

──────────────────────────────────────────────────
  Average Accuracy (AB)   : 0.9252
  Average Macro-F1        : 0.6696
  Forgetting (F)          : 0.0696
  Average AUROC           : 0.9955
──────────────

## Run — Ablation Studies (NEW: Table 4 equivalent)

In [49]:
# ARC pipeline results se variables reconstruct karo
CFG_ATC_ARC = dict(CFG, arc_epsilon=atc_best_eps, arc_theta=atc_best_theta)
CFG_DB_ARC  = dict(CFG, arc_epsilon=db_best_eps,  arc_theta=db_best_theta)
print(f'ATC ARC thresholds: ε={atc_best_eps}  θ={atc_best_theta}')
print(f'DB  ARC thresholds: ε={db_best_eps}   θ={db_best_theta}')

ATC ARC thresholds: ε=0.8  θ=0.02
DB  ARC thresholds: ε=0.8   θ=0.02


In [50]:
# Manually set karo sweep ke best values ya default
atc_best_eps   = 0.8
atc_best_theta = 0.2
db_best_eps    = 0.8
db_best_theta  = 0.2

CFG_ATC_ARC = dict(CFG, arc_epsilon=atc_best_eps, arc_theta=atc_best_theta)
CFG_DB_ARC  = dict(CFG, arc_epsilon=db_best_eps,  arc_theta=db_best_theta)
print(f'ATC ARC thresholds: ε={atc_best_eps}  θ={atc_best_theta}')
print(f'DB  ARC thresholds: ε={db_best_eps}   θ={db_best_theta}')

ATC ARC thresholds: ε=0.8  θ=0.2
DB  ARC thresholds: ε=0.8   θ=0.2


In [51]:
def run_ablation_study(
    tasks, X_tr, y_tr, X_val, y_val, X_te, y_te,
    feat_dim, label_map, cfg, device, dataset_name='Dataset'
):
    print(f'\n{"="*60}')
    print(f'  ABLATION STUDY — {dataset_name}')
    print(f'{"="*60}')

    clf_base = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)
    clf_checkpoints = {}

    print('\nTraining base classifier (shared across all ablation conditions)...')
    for task in tasks:
        tid      = task['task_id']
        seen_cls = task['all_classes']
        print(f'  Task {tid}: {list(task["class_names"].values())}')
        if tid > 0:
            clf_base.grow(len(task['new_classes']), device)
        clf_base = train_classifier_on_task(
            clf_base, X_tr, y_tr, X_val, y_val,
            seen_cls, cfg['clf_epochs'], cfg['clf_lr'], cfg['clf_batch'], device,
            es_patience=cfg.get('es_patience', 10),
            es_min_delta=cfg.get('es_min_delta', 1e-4),
        )
        clf_checkpoints[tid] = copy.deepcopy(clf_base.state_dict())

    conditions = [
        ('Baseline',        False, 'full'),
        ('Retention-only',  True,  'retention'),
        ('Correction-only', True,  'correction'),
        ('Full ARC',        True,  'full'),
    ]

    ablation_rows = []

    for cond_name, use_arc, arc_mode in conditions:
        print(f'\n--- Condition: {cond_name} ---')
        n_tasks   = len(tasks)
        final_tid = n_tasks - 1

        clf_eval = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)
        for task in tasks:
            if task['task_id'] > 0:
                clf_eval.grow(len(task['new_classes']), device)
        clf_eval.load_state_dict(clf_checkpoints[final_tid])
        clf_eval.eval()

        persistent_clf_arc = copy.deepcopy(clf_eval) if use_arc else None

        acc_list, f1_list, auroc_list, forget_list = [], [], [], []
        peak_accs = {}

        # Peak accuracy — tuple unpack FIX
        for task in tasks:
            tid = task['task_id']
            clf_t = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)
            for t2 in tasks[:tid+1]:
                if t2['task_id'] > 0:
                    clf_t.grow(len(t2['new_classes']), device)
            clf_t.load_state_dict(clf_checkpoints[tid])
            clf_t.eval()
            res_peak, _ = evaluate_task(          # ← tuple unpack
                clf_t, X_te, y_te, task['all_classes'],
                task_id=tid, n_cpt=cfg['classes_per_task'],
                epsilon=cfg['arc_epsilon'], theta=cfg['arc_theta'],
                temp=cfg['arc_temp'], arc_lr=cfg['arc_lr'],
                use_arc=False, device=device,
            )
            peak_accs[tid] = res_peak['accuracy']

        # Final task eval — tuple unpack FIX
        for prev_task in tasks:
            prev_tid  = prev_task['task_id']
            prev_seen = prev_task['all_classes']

            res, persistent_clf_arc = evaluate_task(   # ← tuple unpack + state carry
                clf_eval, X_te, y_te, prev_seen,
                task_id            = final_tid,
                n_cpt              = cfg['classes_per_task'],
                epsilon            = cfg['arc_epsilon'],
                theta              = cfg['arc_theta'],
                temp               = cfg['arc_temp'],
                arc_lr             = cfg['arc_lr'],
                use_arc            = use_arc,
                arc_mode           = arc_mode,
                device             = device,
                persistent_clf_arc = persistent_clf_arc,
            )
            acc_list.append(res['accuracy'])
            f1_list.append(res['macro_f1'])
            auroc_list.append(res['auroc'])
            if prev_tid < final_tid:
                forget_list.append(peak_accs[prev_tid] - res['accuracy'])

            print(f"  Task {prev_tid}: acc={res['accuracy']:.4f}  f1={res['macro_f1']:.4f}")

        avg_acc    = float(np.mean(acc_list))
        avg_f1     = float(np.mean(f1_list))
        forgetting = float(np.mean(forget_list)) if forget_list else 0.0
        avg_auroc  = float(np.nanmean(auroc_list))

        print(f'  → AB={avg_acc:.4f}  Macro-F1={avg_f1:.4f}  F={forgetting:.4f}  AUROC={avg_auroc:.4f}')

        ablation_rows.append({
            'Dataset'  : dataset_name,
            'Condition': cond_name,
            'AB'       : round(avg_acc,    4),
            'Macro-F1' : round(avg_f1,     4),
            'F'        : round(forgetting, 4),
            'AUROC'    : round(avg_auroc,  4),
        })

    ablation_df = pd.DataFrame(ablation_rows)
    baseline_ab = ablation_df.loc[ablation_df['Condition'] == 'Baseline', 'AB'].values[0]
    baseline_f1 = ablation_df.loc[ablation_df['Condition'] == 'Baseline', 'Macro-F1'].values[0]
    ablation_df['ΔAB']       = (ablation_df['AB']       - baseline_ab).round(4)
    ablation_df['ΔMacro-F1'] = (ablation_df['Macro-F1'] - baseline_f1).round(4)

    print(f'\n{"─"*60}')
    print(f'  ABLATION SUMMARY — {dataset_name}')
    print(f'{"─"*60}')
    print(ablation_df.to_string(index=False))

    return ablation_df


print('Ablation study defined (tuple unpack fixed)')

Ablation study defined (tuple unpack fixed)


In [52]:
print('Running DrugBank Ablation Study...')
db_ablation_df = run_ablation_study(
    db_tasks, db_X_tr, db_y_tr, db_X_val, db_y_val, db_X_te, db_y_te,
    FEAT_DIM, db_label_map, CFG_DB_ARC, DEVICE, dataset_name='atc'
)

Running DrugBank Ablation Study...

  ABLATION STUDY — atc

Training base classifier (shared across all ablation conditions)...
  Task 0: ['carbonyl', 'nitro']
  Epoch  10/100  loss=0.0269  val_macro_f1=0.916  patience=0/10
  Epoch  20/100  loss=0.0121  val_macro_f1=0.832  patience=10/10
  [Early Stop] Stopped at epoch 20. Best val_macro_f1=0.9158
  Task 1: ['carbonyl', 'nitro', 'none_detected', 'sulfonyl']
  Epoch  10/100  loss=0.1282  val_macro_f1=0.835  patience=2/10
  Epoch  20/100  loss=0.0946  val_macro_f1=0.883  patience=1/10
  [Early Stop] Stopped at epoch 29. Best val_macro_f1=0.8840

--- Condition: Baseline ---
  Task 0: acc=0.9235  f1=0.6467
  Task 1: acc=0.9294  f1=0.7273
  → AB=0.9264  Macro-F1=0.6870  F=0.0678  AUROC=0.9957

--- Condition: Retention-only ---
    OTD [mode=retention]: retention=543/575 correction=0/575 current=32/575  (batch_theta=1.200)
  Task 0: acc=0.9896  f1=0.8307
    OTD [mode=retention]: retention=756/793 correction=2/793 current=35/793  (batch_thet

In [53]:
print('Running DrugBank Ablation Study...')
db_ablation_df = run_ablation_study(
    db_tasks, db_X_tr, db_y_tr, db_X_val, db_y_val, db_X_te, db_y_te,
    FEAT_DIM, db_label_map, CFG_DB_ARC, DEVICE, dataset_name='DrugBank'
)

Running DrugBank Ablation Study...

  ABLATION STUDY — DrugBank

Training base classifier (shared across all ablation conditions)...
  Task 0: ['carbonyl', 'nitro']
  Epoch  10/100  loss=0.0286  val_macro_f1=0.810  patience=8/10
  Epoch  20/100  loss=0.0112  val_macro_f1=0.862  patience=4/10
  [Early Stop] Stopped at epoch 26. Best val_macro_f1=0.8991
  Task 1: ['carbonyl', 'nitro', 'none_detected', 'sulfonyl']
  Epoch  10/100  loss=0.1256  val_macro_f1=0.870  patience=0/10
  Epoch  20/100  loss=0.0872  val_macro_f1=0.873  patience=1/10
  Epoch  30/100  loss=0.0701  val_macro_f1=0.877  patience=1/10
  [Early Stop] Stopped at epoch 39. Best val_macro_f1=0.8996

--- Condition: Baseline ---
  Task 0: acc=0.9513  f1=0.6875
  Task 1: acc=0.9496  f1=0.7654
  → AB=0.9504  Macro-F1=0.7265  F=0.0400  AUROC=0.9957

--- Condition: Retention-only ---
    OTD [mode=retention]: retention=545/575 correction=0/575 current=30/575  (batch_theta=1.200)
  Task 0: acc=0.9896  f1=0.8307
    OTD [mode=retent

In [56]:
print('Running ATC Ablation Study...')
atc_ablation_df = run_ablation_study(
    atc_tasks, atc_X_tr, atc_y_tr, atc_X_val, atc_y_val, atc_X_te, atc_y_te,
    FEAT_DIM, atc_label_map, CFG_ATC_ARC, DEVICE, dataset_name='ATC'
)

print('Running DrugBank Ablation Study...')
db_ablation_df = run_ablation_study(
    db_tasks, db_X_tr, db_y_tr, db_X_val, db_y_val, db_X_te, db_y_te,
    FEAT_DIM, db_label_map, CFG_DB_ARC, DEVICE, dataset_name='DrugBank'
)

# Combined table
combined_ablation = pd.concat([atc_ablation_df, db_ablation_df], ignore_index=True)
print('\n=== COMBINED ABLATION TABLE (Table 4 equivalent) ===')
print(combined_ablation.to_string(index=False))

Running ATC Ablation Study...

  ABLATION STUDY — ATC

Training base classifier (shared across all ablation conditions)...
  Task 0: ['ATC-0', 'ATC-1']
  Epoch  10/100  loss=0.3552  val_macro_f1=0.416  patience=2/10
  Epoch  20/100  loss=0.2650  val_macro_f1=0.438  patience=2/10
  [Early Stop] Stopped at epoch 28. Best val_macro_f1=0.4699
  Task 1: ['ATC-0', 'ATC-1', 'ATC-2', 'ATC-3']
  Epoch  10/100  loss=0.7487  val_macro_f1=0.284  patience=1/10
  Epoch  20/100  loss=0.6184  val_macro_f1=0.307  patience=0/10
  Epoch  30/100  loss=0.5549  val_macro_f1=0.277  patience=2/10
  [Early Stop] Stopped at epoch 38. Best val_macro_f1=0.3302
  Task 2: ['ATC-0', 'ATC-1', 'ATC-2', 'ATC-3', 'ATC-4', 'ATC-5']
  Epoch  10/100  loss=0.7765  val_macro_f1=0.353  patience=3/10
  Epoch  20/100  loss=0.6699  val_macro_f1=0.383  patience=1/10
  [Early Stop] Stopped at epoch 29. Best val_macro_f1=0.4199
  Task 3: ['ATC-0', 'ATC-1', 'ATC-2', 'ATC-3', 'ATC-4', 'ATC-5', 'ATC-6', 'ATC-7']
  Epoch  10/100  loss=

## Compute BWT / FWT

In [61]:
# ── ATC: saare 3 conditions ──────────────────────────────────────
atc_res_replay, atc_AB_replay, atc_F1_replay, atc_F_replay, atc_AUROC_replay, atc_ckpts_replay = run_cil_pipeline(
    tasks=atc_tasks, X_tr=atc_X_tr, y_tr=atc_y_tr,
    X_val=atc_X_val, y_val=atc_y_val, X_te=atc_X_te, y_te=atc_y_te,
    feat_dim=FEAT_DIM, label_map=atc_label_map,
    cfg=CFG, device=DEVICE, use_arc=False, name='ATC Replay-Baseline'
)

atc_res_noreplay, atc_AB_norepay, atc_F1_norepay, atc_F_norepay, atc_AUROC_norepay, atc_ckpts_norepay = run_cil_pipeline_no_replay(
    tasks=atc_tasks, X_tr=atc_X_tr, y_tr=atc_y_tr,
    X_val=atc_X_val, y_val=atc_y_val, X_te=atc_X_te, y_te=atc_y_te,
    feat_dim=FEAT_DIM, label_map=atc_label_map,
    cfg=CFG, device=DEVICE, name='ATC No-Replay-Baseline'
)

CFG_ATC_ARC = dict(CFG, arc_epsilon=atc_best_eps, arc_theta=atc_best_theta)
atc_res_arc, atc_AB_arc, atc_F1_arc, atc_F_arc, atc_AUROC_arc, atc_ckpts_arc = run_cil_pipeline(
    tasks=atc_tasks, X_tr=atc_X_tr, y_tr=atc_y_tr,
    X_val=atc_X_val, y_val=atc_y_val, X_te=atc_X_te, y_te=atc_y_te,
    feat_dim=FEAT_DIM, label_map=atc_label_map,
    cfg=CFG_ATC_ARC, device=DEVICE, use_arc=True, name='ATC + ARC'
)

# ── DrugBank: saare 3 conditions ─────────────────────────────────
db_res_replay, db_AB_replay, db_F1_replay, db_F_replay, db_AUROC_replay, db_ckpts_replay = run_cil_pipeline(
    tasks=db_tasks, X_tr=db_X_tr, y_tr=db_y_tr,
    X_val=db_X_val, y_val=db_y_val, X_te=db_X_te, y_te=db_y_te,
    feat_dim=FEAT_DIM, label_map=db_label_map,
    cfg=CFG, device=DEVICE, use_arc=False, name='DrugBank Replay-Baseline'
)

db_res_noreplay, db_AB_norepay, db_F1_norepay, db_F_norepay, db_AUROC_norepay, db_ckpts_norepay = run_cil_pipeline_no_replay(
    tasks=db_tasks, X_tr=db_X_tr, y_tr=db_y_tr,
    X_val=db_X_val, y_val=db_y_val, X_te=db_X_te, y_te=db_y_te,
    feat_dim=FEAT_DIM, label_map=db_label_map,
    cfg=CFG, device=DEVICE, name='DrugBank No-Replay-Baseline'
)

CFG_DB_ARC = dict(CFG, arc_epsilon=db_best_eps, arc_theta=db_best_theta)
db_res_arc, db_AB_arc, db_F1_arc, db_F_arc, db_AUROC_arc, db_ckpts_arc = run_cil_pipeline(
    tasks=db_tasks, X_tr=db_X_tr, y_tr=db_y_tr,
    X_val=db_X_val, y_val=db_y_val, X_te=db_X_te, y_te=db_y_te,
    feat_dim=FEAT_DIM, label_map=db_label_map,
    cfg=CFG_DB_ARC, device=DEVICE, use_arc=True, name='DrugBank + ARC'
)

# ── BWT/FWT ───────────────────────────────────────────────────────
atc_BWT_replay,  atc_FWT_replay  = compute_bwt_fwt_from_checkpoints(
    atc_tasks, atc_X_te, atc_y_te, FEAT_DIM, atc_ckpts_replay, CFG, DEVICE, use_arc=False)
atc_BWT_norepay, atc_FWT_norepay = compute_bwt_fwt_from_checkpoints(
    atc_tasks, atc_X_te, atc_y_te, FEAT_DIM, atc_ckpts_norepay, CFG, DEVICE, use_arc=False)
atc_BWT_arc,     atc_FWT_arc     = compute_bwt_fwt_from_checkpoints(
    atc_tasks, atc_X_te, atc_y_te, FEAT_DIM, atc_ckpts_arc, CFG_ATC_ARC, DEVICE, use_arc=True)

db_BWT_replay,  db_FWT_replay  = compute_bwt_fwt_from_checkpoints(
    db_tasks, db_X_te, db_y_te, FEAT_DIM, db_ckpts_replay, CFG, DEVICE, use_arc=False)
db_BWT_norepay, db_FWT_norepay = compute_bwt_fwt_from_checkpoints(
    db_tasks, db_X_te, db_y_te, FEAT_DIM, db_ckpts_norepay, CFG, DEVICE, use_arc=False)
db_BWT_arc,     db_FWT_arc     = compute_bwt_fwt_from_checkpoints(
    db_tasks, db_X_te, db_y_te, FEAT_DIM, db_ckpts_arc, CFG_DB_ARC, DEVICE, use_arc=True)

print('All done!')


  ATC Replay-Baseline CIL Pipeline  |  ARC=False  arc_mode=full
  [Replay ON]  — Baseline uses experience replay

-- Task 0 | Classes: ['ATC-0', 'ATC-1'] --
  Epoch  10/100  loss=0.3775  val_macro_f1=0.439  patience=0/10
  Epoch  20/100  loss=0.2577  val_macro_f1=0.416  patience=5/10
  [Early Stop] Stopped at epoch 25. Best val_macro_f1=0.4591
  Eval Task 0 → acc=0.7377  macro_f1=0.5564  auroc=0.7457

-- Task 1 | Classes: ['ATC-0', 'ATC-1', 'ATC-2', 'ATC-3'] --
  Epoch  10/100  loss=0.4706  val_macro_f1=0.255  patience=0/10
  Epoch  20/100  loss=0.3756  val_macro_f1=0.263  patience=9/10
  [Early Stop] Stopped at epoch 21. Best val_macro_f1=0.2779
  Eval Task 0 → acc=0.5902  macro_f1=0.5157  auroc=0.7714
  Eval Task 1 → acc=0.4490  macro_f1=0.3884  auroc=0.6926

-- Task 2 | Classes: ['ATC-0', 'ATC-1', 'ATC-2', 'ATC-3', 'ATC-4', 'ATC-5'] --
  Epoch  10/100  loss=0.5822  val_macro_f1=0.388  patience=1/10
  Epoch  20/100  loss=0.4697  val_macro_f1=0.364  patience=2/10
  Epoch  30/100  los

## Final Comparison Table

In [62]:
comparison = pd.DataFrame([
    {'Dataset': 'ATC', 'Method': 'Baseline (replay)',
     'AB': round(atc_AB_replay,  4), 'Macro-F1': round(atc_F1_replay,  4),
     'F':  round(atc_F_replay,   4), 'AUROC':    round(atc_AUROC_replay, 4),
     'BWT': round(atc_BWT_replay, 4), 'FWT': round(atc_FWT_replay, 4)},
    {'Dataset': 'ATC', 'Method': 'Baseline (no-replay)',
     'AB': round(atc_AB_norepay, 4), 'Macro-F1': round(atc_F1_norepay, 4),
     'F':  round(atc_F_norepay,  4), 'AUROC':    round(atc_AUROC_norepay, 4),
     'BWT': round(atc_BWT_norepay, 4), 'FWT': round(atc_FWT_norepay, 4)},
    {'Dataset': 'ATC', 'Method': f'+ ARC (ε={atc_best_eps} θ={atc_best_theta})',
     'AB': round(atc_AB_arc,    4), 'Macro-F1': round(atc_F1_arc,    4),
     'F':  round(atc_F_arc,     4), 'AUROC':    round(atc_AUROC_arc,  4),
     'BWT': round(atc_BWT_arc,  4), 'FWT': round(atc_FWT_arc, 4)},
    {'Dataset': 'DrugBank', 'Method': 'Baseline (replay)',
     'AB': round(db_AB_replay,  4), 'Macro-F1': round(db_F1_replay,  4),
     'F':  round(db_F_replay,   4), 'AUROC':    round(db_AUROC_replay, 4),
     'BWT': round(db_BWT_replay, 4), 'FWT': round(db_FWT_replay, 4)},
    {'Dataset': 'DrugBank', 'Method': 'Baseline (no-replay)',
     'AB': round(db_AB_norepay, 4), 'Macro-F1': round(db_F1_norepay, 4),
     'F':  round(db_F_norepay,  4), 'AUROC':    round(db_AUROC_norepay, 4),
     'BWT': round(db_BWT_norepay, 4), 'FWT': round(db_FWT_norepay, 4)},
    {'Dataset': 'DrugBank', 'Method': f'+ ARC (ε={db_best_eps} θ={db_best_theta})',
     'AB': round(db_AB_arc,    4), 'Macro-F1': round(db_F1_arc,    4),
     'F':  round(db_F_arc,     4), 'AUROC':    round(db_AUROC_arc,  4),
     'BWT': round(db_BWT_arc,  4), 'FWT': round(db_FWT_arc, 4)},
])

print('=' * 90)
print('                         FINAL COMPARISON TABLE (v12)')
print('=' * 90)
print(comparison.to_string(index=False))
print()
print('ARC vs No-Replay-Baseline (apples-to-apples):')
print(f'  ATC      → ΔAB={atc_AB_arc-atc_AB_norepay:+.4f}  ΔMacro-F1={atc_F1_arc-atc_F1_norepay:+.4f}  ΔF={atc_F_arc-atc_F_norepay:+.4f}  ΔAUROC={atc_AUROC_arc-atc_AUROC_norepay:+.4f}')
print(f'  DrugBank → ΔAB={db_AB_arc-db_AB_norepay:+.4f}  ΔMacro-F1={db_F1_arc-db_F1_norepay:+.4f}  ΔF={db_F_arc-db_F_norepay:+.4f}  ΔAUROC={db_AUROC_arc-db_AUROC_norepay:+.4f}')
print('=' * 90)
comparison

                         FINAL COMPARISON TABLE (v12)
 Dataset               Method     AB  Macro-F1      F  AUROC     BWT    FWT
     ATC    Baseline (replay) 0.3393    0.3049 0.1190 0.7411 -0.1190 0.2591
     ATC Baseline (no-replay) 0.3443    0.3215 0.1624 0.7431 -0.1624 0.2975
     ATC  + ARC (ε=0.8 θ=0.2) 0.3447    0.3096 0.0706 0.7366 -0.0706 0.2144
DrugBank    Baseline (replay) 0.9200    0.6696 0.0730 0.9954 -0.0730 0.6718
DrugBank Baseline (no-replay) 0.9429    0.7262 0.0487 0.9960 -0.0487 0.6933
DrugBank  + ARC (ε=0.8 θ=0.2) 0.8729    0.7410 0.0000 0.9752  0.0000 0.5079

ARC vs No-Replay-Baseline (apples-to-apples):
  ATC      → ΔAB=+0.0004  ΔMacro-F1=-0.0119  ΔF=-0.0919  ΔAUROC=-0.0065
  DrugBank → ΔAB=-0.0701  ΔMacro-F1=+0.0149  ΔF=-0.0487  ΔAUROC=-0.0207


,Dataset,Method,AB,Macro-F1,F,AUROC,BWT,FWT
0,ATC,Baseline (replay),0.3393,0.3049,0.1190,0.7411,-0.1190,0.2591
1,ATC,Baseline (no-replay),0.3443,0.3215,0.1624,0.7431,-0.1624,0.2975
2,ATC,+ ARC (ε=0.8 θ=0.2),0.3447,0.3096,0.0706,0.7366,-0.0706,0.2144
3,DrugBank,Baseline (replay),0.9200,0.6696,0.0730,0.9954,-0.0730,0.6718
4,DrugBank,Baseline (no-replay),0.9429,0.7262,0.0487,0.9960,-0.0487,0.6933
5,DrugBank,+ ARC (ε=0.8 θ=0.2),0.8729,0.7410,0.0000,0.9752,0.0000,0.5079


## Save Results

In [ ]:
OUT = CFG['output_dir']

atc_res_replay.to_csv   (os.path.join(OUT, 'atc_replay_baseline.csv'),    index=False)
atc_res_noreplay.to_csv  (os.path.join(OUT, 'atc_norepay_baseline.csv'),   index=False)
atc_res_arc.to_csv      (os.path.join(OUT, 'atc_arc.csv'),                index=False)
db_res_replay.to_csv    (os.path.join(OUT, 'db_replay_baseline.csv'),     index=False)
db_res_norepay.to_csv   (os.path.join(OUT, 'db_norepay_baseline.csv'),    index=False)
db_res_arc.to_csv       (os.path.join(OUT, 'db_arc.csv'),                 index=False)
comparison.to_csv        (os.path.join(OUT, 'comparison_v12.csv'),         index=False)
combined_ablation.to_csv (os.path.join(OUT, 'ablation_study_v12.csv'),    index=False)
atc_sweep_df.to_csv      (os.path.join(OUT, 'atc_threshold_sweep.csv'),    index=False)
db_sweep_df.to_csv       (os.path.join(OUT, 'db_threshold_sweep.csv'),     index=False)

meta = {
    'atc': {
        'best_eps': atc_best_eps, 'best_theta': atc_best_theta,
        'replay_baseline' : {'AB': atc_AB_replay,  'F1': atc_F1_replay,  'F': atc_F_replay},
        'norepay_baseline': {'AB': atc_AB_norepay, 'F1': atc_F1_norepay, 'F': atc_F_norepay},
        'arc'             : {'AB': atc_AB_arc,     'F1': atc_F1_arc,     'F': atc_F_arc},
    },
    'drugbank': {
        'best_eps': db_best_eps, 'best_theta': db_best_theta,
        'replay_baseline' : {'AB': db_AB_replay,  'F1': db_F1_replay,  'F': db_F_replay},
        'norepay_baseline': {'AB': db_AB_norepay, 'F1': db_F1_norepay, 'F': db_F_norepay},
        'arc'             : {'AB': db_AB_arc,     'F1': db_F1_arc,     'F': db_F_arc},
    },
    'config': CFG,
    'fixes_applied': [
        'FIX-1 through FIX-17: inherited from v11',
        'FIX-18: compute_batch_theta uses min() not max() — adaptive can now LOWER theta',
        'FIX-19: evaluate_task receives task_id=tid (current training task) not prev_tid',
        'FIX-20: sweep_arc_thresholds resets clf_arc from checkpoint per (eps,theta) combo',
        'NEW: run_ablation_study() — Table 4 equivalent for Retention-only / Correction-only / Full ARC',
    ]
}
with open(os.path.join(OUT, 'metadata_v12.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print('All results saved to:', OUT)